# Basic Machine Learning with Rust

This notebook demonstrates machine learning concepts using Rust with evcxr.

The Python version uses:
- pandas for data manipulation
- numpy for numerical operations
- matplotlib for visualization
- scikit-learn for machine learning

This Rust version demonstrates equivalent concepts using basic Rust types.

## 1. Import Libraries

In [2]:
:dep rand = "0.8"

In [3]:
use rand::Rng;

## 2. Complete ML Workflow

Due to evcxr's state management limitations, we'll combine the entire workflow in one cell.

In [4]:
// Generate synthetic dataset
let n_samples: usize = 100;
let n_features: usize = 2;

let mut x: Vec<(f64, f64)> = Vec::new();
let mut y: Vec<i32> = Vec::new();

for _ in 0..n_samples {
    let f1: f64 = rand::random::<f64>() * 20.0 - 10.0;
    let f2: f64 = rand::random::<f64>() * 20.0 - 10.0;
    x.push((f1, f2));
    y.push(if f1 + f2 > 0.0 { 1 } else { 0 });
}

println!("Dataset shape: ({}, {})", n_samples, n_features);
println!("Class 0: {}", y.iter().filter(|&&v| v == 0).count());
println!("Class 1: {}", y.iter().filter(|&&v| v == 1).count());

// Train/test split
let split_point: usize = (n_samples * 8) / 10;
let mut x_train: Vec<(f64, f64)> = Vec::new();
let mut x_test: Vec<(f64, f64)> = Vec::new();
let mut y_train: Vec<i32> = Vec::new();
let mut y_test: Vec<i32> = Vec::new();

for i in 0..split_point {
    x_train.push(x[i]);
    y_train.push(y[i]);
}
for i in split_point..n_samples {
    x_test.push(x[i]);
    y_test.push(y[i]);
}

println!("\nTraining set size: {}", x_train.len());
println!("Test set size: {}", x_test.len());

// Train logistic regression
let mut w0: f64 = rand::random::<f64>() - 0.5;
let mut w1: f64 = rand::random::<f64>() - 0.5;
let mut bias: f64 = 0.0;

let epochs: usize = 1000;
let learning_rate: f64 = 0.01;
let n: usize = x_train.len();

for _ in 0..epochs {
    let (mut gw0, mut gw1, mut gb): (f64, f64, f64) = (0.0, 0.0, 0.0);
    for i in 0..n {
        let z: f64 = w0 * x_train[i].0 + w1 * x_train[i].1 + bias;
        let pred: f64 = 1.0 / (1.0 + (-z).exp());
        let error: f64 = pred - y_train[i] as f64;
        gb += error;
        gw0 += error * x_train[i].0;
        gw1 += error * x_train[i].1;
    }
    w0 -= learning_rate * gw0 / n as f64;
    w1 -= learning_rate * gw1 / n as f64;
    bias -= learning_rate * gb / n as f64;
}

println!("\nModel trained successfully!");
println!("Weights: w0={:.4}, w1={:.4}, bias={:.4}", w0, w1, bias);

// Make predictions
let mut y_pred: Vec<i32> = Vec::new();
for i in 0..x_test.len() {
    let z: f64 = w0 * x_test[i].0 + w1 * x_test[i].1 + bias;
    let pred: f64 = 1.0 / (1.0 + (-z).exp());
    y_pred.push(if pred > 0.5 { 1 } else { 0 });
}

// Calculate accuracy
let mut correct: usize = 0;
for i in 0..y_test.len() {
    if y_test[i] == y_pred[i] {
        correct += 1;
    }
}
let accuracy: f64 = correct as f64 / y_test.len() as f64;

println!("\nAccuracy: {:.4}", accuracy);

// Confusion matrix
let mut cm: [[usize; 2]; 2] = [[0; 2]; 2];
for i in 0..y_test.len() {
    cm[y_test[i] as usize][y_pred[i] as usize] += 1;
}

println!("\nConfusion Matrix:");
println!("            Predicted");
println!("       |    0    |    1    |");
println!("Actual 0 |   {}    |   {}    |", cm[0][0], cm[0][1]);
println!("       1 |   {}    |   {}    |", cm[1][0], cm[1][1]);

// Feature importance
println!("\nFeature importance (absolute weights):");
println!("  feature_0: {:.4}", w0.abs());
println!("  feature_1: {:.4}", w1.abs());

Dataset shape: (100, 2)
Class 0: 53
Class 1: 47

Training set size: 80
Test set size: 20

Model trained successfully!
Weights: w0=0.9672, w1=1.0208, bias=0.0418

Accuracy: 0.9500

Confusion Matrix:
            Predicted
       |    0    |    1    |
Actual 0 |   9    |   1    |
       1 |   0    |   10    |

Feature importance (absolute weights):
  feature_0: 0.9672
  feature_1: 1.0208


## 3. Summary

This notebook demonstrated:
1. Synthetic data generation (like `make_classification`)
2. Train/test split (like `train_test_split`)
3. Logistic regression from scratch (like `LogisticRegression`)
4. Model evaluation with accuracy
5. Confusion matrix (like `confusion_matrix`)
6. Feature importance analysis (like model coefficients)

Note: Due to evcxr's limitations with variable state across cells, the entire workflow is combined in a single cell.